In [1]:
import pandas as pd
import numpy as np
import librosa

In [2]:
train_df = pd.read_pickle('train_embedded.pkl')
test_df = pd.read_pickle("test_embedded.pkl")
train_df.head()

,signal_data,language,speaker,label,gender,c,length,w2v_base_emb_pooled,w2v_base_emb,w2v_xlr_emb,w2v_xlr_emb_pooled,mel_spec,mel_spec_padded,mel_spec_pooled
176,"[0.000152587890625, 0.000823974609375, 0.00082...",SA,A,4,F,c,9501,"[-0.015912058, 0.23642024, 0.12297839, 0.21679...","[[0.031614933, 0.27669638, 0.15805711, 0.19895...","[[-0.33289763, -0.16397952, -0.044866655, 0.08...","[-0.043815523, -0.08047871, 0.052105945, 0.042...","[[[-55.064064, -53.257996, -59.722855, -48.224...","[[[-55.064064, -53.257996, -59.722855, -48.224...","[-50.13206, -48.11771, -48.631584, -45.271175,..."
165,"[-3.0517578125e-05, 0.0, 0.0, 0.0, 3.051757812...",FR,F,2,F,c,6201,"[0.38915816, 0.3792101, 0.090889215, 0.4901088...","[[0.40043145, 0.3911669, 0.12556341, 0.4663496...","[[-0.28900322, -0.17436437, -0.04185667, 0.092...","[-0.04911978, -0.11596217, 0.06338483, 0.03236...","[[[-64.447174, -65.83354, -78.590675, -77.9472...","[[[-64.447174, -65.83354, -78.590675, -77.9472...","[-57.80567, -54.742844, -53.918205, -49.423088..."
126,"[9.1552734375e-05, 0.000152587890625, 0.000244...",FR,B,3,F,c,10001,"[0.17522134, 0.23353295, -0.1157971, 0.3438407...","[[0.4023691, 0.1572093, 0.19292863, 0.04548339...","[[-0.25203943, -0.1384559, -0.065831274, 0.049...","[-0.01860454, -0.07754524, 0.047171906, 0.0199...","[[[-79.21712, -73.76115, -75.72417, -74.95683,...","[[[-79.21712, -73.76115, -75.72417, -74.95683,...","[-66.945656, -60.16159, -57.949112, -54.54152,..."
103,"[0.0, 0.0, 0.0, -0.0078125, 0.0, -0.0078125, -...",SI,C,9,M,c,9001,"[0.1901748, 0.27533138, -0.021832537, 0.112002...","[[0.33527187, 0.33407038, 0.18568435, 0.031803...","[[-0.22652705, -0.19961871, -0.058231324, 0.10...","[-0.0071312133, -0.13924864, 0.030800128, 0.02...","[[[-35.610336, -34.771824, -35.57545, -34.6930...","[[[-35.610336, -34.771824, -35.57545, -34.6930...","[-36.700752, -34.88265, -35.53902, -23.765522,..."
70,"[-0.010894775390625, -0.018310546875, -0.00708...",IT,E,5,F,c,15000,"[0.07980928, 0.08479349, 0.17919528, 0.1372474...","[[0.0787321, 0.28121114, 0.21040522, 0.2661217...","[[-0.25639585, -0.231193, -0.0117138345, 0.057...","[0.050507627, -0.04553041, 0.06576309, 0.01915...","[[[-58.969536, -49.85008, -48.191605, -46.3154...","[[[-58.969536, -49.85008, -48.191605, -46.3154...","[-49.63411, -49.047318, -48.6203, -53.586643, ..."


In [3]:
def get_cnn_mfcc_librosa(signal, sr=16000, n_mfcc=20):
    if isinstance(signal, list):
        signal = np.array(signal, dtype=np.float32)
        
    if signal.size == 0:
        return np.zeros((3, n_mfcc, 1), dtype=np.float32)

    # 1. Pre-emphasis (matching your custom math code!)
    signal_preemph = librosa.effects.preemphasis(signal, coef=0.97)

    # 2. Extract Base MFCCs
    # Notice we use the same n_fft=512 and hop_length=160 as the Mel Spec!
    mfcc = librosa.feature.mfcc(
        y=signal_preemph, 
        sr=sr, 
        n_mfcc=n_mfcc,
        n_fft=512,
        hop_length=160
    )

    # 3. Compute Deltas and Delta-Deltas
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    # 4. Stack into 3 channels (Shape: 3, 20, Time)
    stacked_mfcc = np.stack([mfcc, delta, delta2], axis=0)
    
    return stacked_mfcc.astype(np.float32)

# Extract raw features
train_df['mfcc_raw'] = train_df['signal_data'].apply(lambda x: get_cnn_mfcc_librosa(x, n_mfcc=20))
test_df['mfcc_raw'] = test_df['signal_data'].apply(lambda x: get_cnn_mfcc_librosa(x, n_mfcc=20))

/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 1. Find the maximum time dimension (axis 2) in the training set
train_max = train_df['mfcc_raw'].apply(lambda x: x.shape[2]).max()

# 2. Find the maximum time dimension (axis 2) in the testing set
test_max = test_df['mfcc_raw'].apply(lambda x: x.shape[2]).max()

# 3. Calculate the absolute global maximum
global_max_len = max(train_max, test_max)

print(f"The global maximum length is: {global_max_len} frames")

# # 4. Apply the padding using the dynamic length
# train_df['mfcc_padded'] = train_df['mfcc_raw'].apply(lambda x: pad_3d_mfcc(x, global_max_len))
# test_df['mfcc_padded'] = test_df['mfcc_raw'].apply(lambda x: pad_3d_mfcc(x, global_max_len))

The global maximum length is: 153 frames


In [6]:
def pad_3d_mfcc(stacked_mfcc, max_len):
    current_time = stacked_mfcc.shape[2]
    
    if current_time < max_len:
        pad_width = max_len - current_time
        
        # Pad all 3 channels with 0 
        return np.pad(
            stacked_mfcc, 
            pad_width=((0, 0), (0, 0), (0, pad_width)), 
            mode='constant', 
            constant_values=0
        )
    
    # Crop if too long
    return stacked_mfcc[:, :, :max_len]

global_max_len = 160

# Apply the padding
train_df['mfcc_padded'] = train_df['mfcc_raw'].apply(lambda x: pad_3d_mfcc(x, global_max_len))
test_df['mfcc_padded'] = test_df['mfcc_raw'].apply(lambda x: pad_3d_mfcc(x, global_max_len))

In [7]:
def extract_pooled_mfcc_from_3d(stacked_mfcc):
    features = [
        np.mean(stacked_mfcc[0], axis=1), np.std(stacked_mfcc[0], axis=1),
        np.mean(stacked_mfcc[1], axis=1), np.std(stacked_mfcc[1], axis=1),
        np.mean(stacked_mfcc[2], axis=1), np.std(stacked_mfcc[2], axis=1)
    ]
    return np.concatenate(features)

# Remember to use the UNPADDED 'mfcc_raw' here!
train_df['mfcc_pooled'] = train_df['mfcc_raw'].apply(extract_pooled_mfcc_from_3d)
test_df['mfcc_pooled'] = test_df['mfcc_raw'].apply(extract_pooled_mfcc_from_3d)

In [11]:
print(train_df['mfcc_pooled'].iloc[0].shape)
print(train_df['mfcc_padded'].iloc[0].shape)
print(test_df['mfcc_pooled'].iloc[0].shape)
print(test_df['mfcc_padded'].iloc[0].shape)

(120,)
(3, 20, 160)
(120,)
(3, 20, 160)


In [12]:
# Save the training set
train_df.to_pickle("train_embedded.pkl")

# Save the testing set
test_df.to_pickle("test_embedded.pkl")